# SP500 Executive Role Classification — Fine-Tune Gemma 4 31B

Based on the [official Unsloth Gemma 4 notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Gemma4_(31B)-Text.ipynb), adapted for SP500 executive classification data.

**Runtime**: A100 40GB+ recommended. Go to Runtime → Change runtime type → A100.

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, '0.0.34')
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install --no-deps git+https://github.com/huggingface/transformers.git
!pip install torchcodec
import torch; torch._dynamo.config.recompile_limit = 64;

In [ ]:
from unsloth import FastModel
import torch

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-4-31B-it",
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
    full_finetuning = False,
)

In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r = 128,
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    random_state = 42,
)

## Load & Format Training Data

In [ ]:
import json, random, os
from datasets import Dataset
from unsloth.chat_templates import get_chat_template

# Set up Gemma 4 chat template (non-thinking — no reasoning overhead for classification)
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-4",
)

# Clone repo with training data
REPO_DIR = "/content/finetune_sp500"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/daalbert981/finetune_sp500.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

DATA_PATH = os.path.join(REPO_DIR, "Training Datasets", "full_data_n_4977.jsonl")
with open(DATA_PATH, "r") as f:
    examples = [json.loads(line) for line in f]

print(f"Loaded {len(examples)} examples")

# Shuffle and split 95/5
random.seed(42)
random.shuffle(examples)
split_idx = int(len(examples) * 0.95)

train_dataset = Dataset.from_list(examples[:split_idx])
valid_dataset = Dataset.from_list(examples[split_idx:])
print(f"Train: {len(train_dataset)}, Valid: {len(valid_dataset)}")

In [ ]:
def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [
        tokenizer.apply_chat_template(
            convo, tokenize=False, add_generation_prompt=False
        ).removeprefix('<bos>')
        for convo in convos
    ]
    return {"text": texts}

train_dataset = train_dataset.map(formatting_prompts_func, batched=True)
valid_dataset = valid_dataset.map(formatting_prompts_func, batched=True)

print(f"Train samples: {len(train_dataset)}, Valid samples: {len(valid_dataset)}")
print("=== FULL FORMATTED TEXT ===")
print(train_dataset[0]["text"])

## Train

In [ ]:
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        num_train_epochs = 3,
        warmup_ratio = 0.03,
        learning_rate = 2e-4,
        logging_steps = 10,
        eval_strategy = "no",
        save_strategy = "steps",
        save_steps = 200,
        save_total_limit = 2,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 42,
        output_dir = "outputs",
        report_to = "none",
        bf16 = True,
        max_seq_length = 2048,
    ),
)

# Only train on the model's response, not the prompt
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|turn>user\n",
    response_part = "<|turn>model\n",
)

print(f"Effective batch size: {2 * 4}")
print(f"Train samples: {len(train_dataset)}")
print(f"Approx training steps: ~{len(train_dataset) * 3 // 8}")

In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

In [ ]:
trainer_stats = trainer.train()

In [ ]:
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")

## Quick Inference Test

In [ ]:
messages = [
    {"role": "system", "content": train_dataset[0]["messages"][0]["content"]},
    {"role": "user", "content": "In 2020 the company 'apple inc' had an executive with the name jane doe, whose official role title was: 'senior vice president, human resources'."},
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    return_tensors = "pt",
    tokenize = True,
    return_dict = True,
).to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens = 200,
    temperature = 1.0,
    top_p = 0.95,
    top_k = 64,
)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

## Save to HuggingFace

In [ ]:
# Load HF token from Colab secrets
from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")

# Save LoRA adapters locally
model.save_pretrained("sp500_exec_gemma4_lora")
tokenizer.save_pretrained("sp500_exec_gemma4_lora")
print("LoRA adapters saved locally.")

In [ ]:
# Push LoRA adapters to HuggingFace
model.push_to_hub("daresearch/sp500-exec-classifier-gemma4-lora", token=HF_TOKEN)
tokenizer.push_to_hub("daresearch/sp500-exec-classifier-gemma4-lora", token=HF_TOKEN)
print("LoRA adapters pushed to https://huggingface.co/daresearch/sp500-exec-classifier-gemma4-lora")

In [ ]:
# Save and push GGUF (Q8_0 — only Q8_0, BF16, F16 supported for Gemma 4)
model.push_to_hub_gguf(
    "daresearch/sp500-exec-classifier-gemma4-gguf",
    tokenizer,
    quantization_method = "Q8_0",
    token = HF_TOKEN,
)
print("GGUF pushed to https://huggingface.co/daresearch/sp500-exec-classifier-gemma4-gguf")